# Merge Research Features

## Objective

Combine price, product, holiday, and weather features into one
order-level dataset.

## Rules

- Keep one row per order_id.
- Preserve all original columns and target labels.
- Reject duplicate identifiers and overlapping feature names.
- Preserve missing values.

## Output

data/processed/research_data_enriched.csv

In [3]:
# Purpose: Locate the project and load all feature tables.

from pathlib import Path

import pandas as pd
from IPython.display import display

current_dir = Path.cwd()

if (current_dir / "data/processed/research_data.csv").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data/processed/research_data.csv").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError("Cannot locate research_data.csv.")

DATA_DIR = PROJECT_ROOT / "data/processed"


def read_order_table(filename):
    path = DATA_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path}")

    table = pd.read_csv(
        path,
        dtype={
            "order_id": "string",
            "customer_id": "string",
        },
    )

    if "order_id" not in table.columns:
        raise ValueError(f"{filename}: missing order_id column.")

    if (
        table["order_id"].isna().any()
        or table["order_id"].str.strip().eq("").any()
        or table["order_id"].duplicated().any()
    ):
        raise ValueError(f"{filename}: invalid or duplicated order_id.")

    return table


original = read_order_table("research_data.csv")
price_product = read_order_table("research_data_price_product.csv")
holidays = read_order_table("order_holiday_features.csv")
weather = read_order_table("order_weather_features.csv")

display(pd.DataFrame([
    {
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
    }
    for name, table in [
        ("original", original),
        ("price_product", price_product),
        ("holidays", holidays),
        ("weather", weather),
    ]
]))

,table,rows,columns
0,original,99441,34
1,price_product,99441,46
2,holidays,99441,4
3,weather,99441,12


In [4]:
# Purpose: Merge additional features while preserving original records.

original_ids = set(original["order_id"])

for name, table in [
    ("price_product", price_product),
    ("holidays", holidays),
    ("weather", weather),
]:
    if set(table["order_id"]) != original_ids:
        raise ValueError(
            f"{name}: order IDs differ from the original dataset."
        )

missing_original_columns = (
    set(original.columns) - set(price_product.columns)
)

if missing_original_columns:
    raise ValueError(
        f"Missing original columns: {sorted(missing_original_columns)}"
    )

# Verify that price-product processing preserved original values.
original_values = original.set_index("order_id").sort_index()

pd.testing.assert_frame_equal(
    price_product.set_index("order_id")[
        original_values.columns
    ].sort_index(),
    original_values,
    check_dtype=False,
)

enriched = price_product.copy()

for name, table in [("holidays", holidays), ("weather", weather)]:
    overlapping_columns = (
        set(enriched.columns) & set(table.columns)
    ) - {"order_id"}

    if overlapping_columns:
        raise ValueError(
            f"{name}: overlapping columns: "
            f"{sorted(overlapping_columns)}"
        )

    enriched = enriched.merge(
        table,
        on="order_id",
        how="left",
        validate="one_to_one",
        sort=False,
    )

# Verify row count, identifiers, original values, and target labels.
assert len(enriched) == len(original)
assert enriched["order_id"].is_unique

pd.testing.assert_frame_equal(
    enriched.set_index("order_id")[
        original_values.columns
    ].sort_index(),
    original_values,
    check_dtype=False,
)

expected_columns = (
    price_product.shape[1]
    + holidays.shape[1] - 1
    + weather.shape[1] - 1
)

assert enriched.shape[1] == expected_columns

print("All merge checks passed.")
print("Final dataset shape:", enriched.shape)
print("Valid target labels:", enriched["is_late"].isin([0, 1]).sum())

All merge checks passed.
Final dataset shape: (99441, 60)
Valid target labels: 96470


In [5]:
# Purpose: Export the enriched dataset and additional-feature missingness.

added_columns = [
    column for column in enriched.columns
    if column not in original.columns
]

missing_summary = pd.DataFrame({
    "feature": added_columns,
    "missing_rows": (
        enriched[added_columns].isna().sum().to_numpy()
    ),
    "missing_percent": (
        enriched[added_columns].isna().mean() * 100
    ).round(3).to_numpy(),
})

display(missing_summary)

output_path = DATA_DIR / "research_data_enriched.csv"

enriched.to_csv(output_path, index=False)

quality_dir = PROJECT_ROOT / "outputs/tables/additional_features"
quality_dir.mkdir(parents=True, exist_ok=True)

missing_summary.to_csv(
    quality_dir / "enriched_missing_summary.csv",
    index=False,
)

print("Dataset saved:", output_path.resolve())
print("Missingness report saved:", quality_dir.resolve())

,feature,missing_rows,missing_percent
0,order_total_value,775,0.779
1,freight_to_items_ratio,775,0.779
2,mean_item_value,775,0.779
3,freight_per_item,775,0.779
4,total_product_weight_g,797,0.801
5,total_product_volume_cm3,791,0.795
6,product_category_count,2226,2.239
7,weight_coverage_ratio,775,0.779
8,volume_coverage_ratio,775,0.779
9,category_coverage_ratio,775,0.779


Dataset saved: C:\Users\LENOVO\Downloads\Group5_STAT3013.R11-main (1)\Group5_STAT3013.R11-main\data\processed\research_data_enriched.csv
Missingness report saved: C:\Users\LENOVO\Downloads\Group5_STAT3013.R11-main (1)\Group5_STAT3013.R11-main\outputs\tables\additional_features
